In [ ]:
import pandas as pd

# Load datasets
employees = pd.read_csv("employees.csv", parse_dates=["hire_date", "exit_date"])
attrition = pd.read_csv("attrition_log.csv", parse_dates=["exit_date"])
engagement = pd.read_csv("engagement.csv", parse_dates=["survey_date"])
performance = pd.read_csv("performance.csv", parse_dates=["review_date"])

# Rename attrition exit_date so it does not clash with employees.exit_date.
attrition_detail = attrition.rename(columns={"exit_date": "attrition_exit_date"})

# Employee-level combined dataset: one row per employee.
# Use this for headcount, department, leaver, salary, and hire-source analysis.
df = employees.merge(
    attrition_detail,
    on="employee_id",
    how="left"
)

# Fully expanded activity dataset for later engagement/performance analysis.
# This has multiple rows per employee because surveys and reviews repeat over time.
combined_activity = (
    df.merge(engagement, on="employee_id", how="left")
      .merge(performance, on="employee_id", how="left")
)

# One row per person who left, joined back to employee fields needed for analysis.
leavers = attrition_detail.merge(
    employees[["employee_id", "department", "hire_source", "salary", "status"]],
    on="employee_id",
    how="left"
).query('status == "departed"')


,employee_id,attrition_exit_date,exit_type,stated_exit_reason,notice_period_served,regrettable_flag,performance_band_at_exit,salary_at_exit,manager_id_at_exit,pathway,department,hire_source,salary,status
0,E00005,2024-12-24,voluntary,Relocation,True,False,Meets Expectations,132400.0,E12337,push,Wealth Management,direct,132400.0,departed
1,E00006,2025-04-17,voluntary,Work-life balance,True,False,Below Expectations,120600.0,E00218,push,Corporate Operations,agency,120600.0,departed
2,E00008,2024-02-19,voluntary,Career advancement,True,False,Meets Expectations,117800.0,E00249,push,Technology,acquisition,117800.0,departed
3,E00021,2024-05-24,voluntary,Career advancement,True,False,Meets Expectations,125500.0,E08861,push,Insurance,referral,125500.0,departed
4,E00023,2025-07-04,voluntary,Career advancement,False,False,Outstanding,175400.0,E09824,pull,Technology,direct,175400.0,departed


In [10]:
# Headcount share vs exit share — the "over-indexed" check for your hook
dept_headcount_share = leavers.drop_duplicates('employee_id')['department'].value_counts(normalize=True) * 100
dept_exit_share = (leavers[leavers['exit_type'] == 'voluntary']
                    .drop_duplicates('employee_id')['department']
                    .value_counts(normalize=True) * 100)

dept_compare = pd.DataFrame({
    'headcount_share_%': dept_headcount_share,
    'voluntary_exit_share_%': dept_exit_share
}).fillna(0)
dept_compare['over_index_ratio'] = dept_compare['voluntary_exit_share_%'] / dept_compare['headcount_share_%']
display(dept_compare.sort_values('over_index_ratio', ascending=False))

# R&C-specific dollar estimate
rc_voluntary_regret = leavers[(leavers['department'] == 'Risk & Compliance') &
                        (leavers['exit_type'] == 'voluntary') &
                        (leavers['regrettable_flag'] == True)].drop_duplicates('employee_id')

rc_headcount = leavers[leavers['department'] == 'Risk & Compliance'].drop_duplicates('employee_id')

n_rc_voluntary_regret_exits = len(rc_voluntary_regret)
avg_rc_salary = rc_headcount['salary'].mean()

# standard rule-of-thumb replacement cost multiplier — state this assumption on the slide
REPLACEMENT_COST_MULTIPLIER = 1.0  # 100% of salary; adjust to 0.5–1.5 based on role seniority if you have it

estimated_rc_cost = n_rc_voluntary_regret_exits * avg_rc_salary * REPLACEMENT_COST_MULTIPLIER

print(f"R&C regrettable voluntary exits: {n_rc_voluntary_regret_exits}")
print(f"R&C avg salary: ${avg_rc_salary:,.0f}")
print(f"Estimated R&C regrettable attrition cost: ${estimated_rc_cost:,.0f}")
print(f"As % of the $22–25M regrettable attrition bucket: {estimated_rc_cost/23_500_000*100:.1f}%")

,headcount_share_%,voluntary_exit_share_%,over_index_ratio
department,,,
Executive Leadership,1.357143,1.500441,1.105588
Corporate Operations,12.500000,12.886143,1.030891
Retail Banking,21.857143,22.241836,1.017600
Risk & Compliance,17.071429,17.210944,1.008172
Technology,22.500000,22.683142,1.008140
Insurance,13.142857,12.621359,0.960321
Wealth Management,11.571429,10.856134,0.938184


R&C regrettable voluntary exits: 33
R&C avg salary: $129,519
Estimated R&C regrettable attrition cost: $4,274,135
As % of the $22–25M regrettable attrition bucket: 18.2%
